# Store DNA — External Data Extractor

This notebook implements the **scrape → extract → structure** pipeline for **Retail StoreDNA**.

It collects **public, store-level signals** from the open web (retailer pages, reviews, social pages, local news, etc.) and uses **Azure OpenAI** to turn raw page text into **structured JSON features** per store.

---

## Pipeline overview

```
Store URL list  →  HTTP scrape  →  Clean text  →  LLM extraction  →  JSON per store  →  (later) embeddings
```

## What this notebook is (and is not)

| ✅ Does | ❌ Does not |
|--------|------------|
| Scrape any **public URL** you provide | Connect to Facebook/Google APIs directly |
| Extract store-relevant attributes via LLM | Replace internal POS / sales data |
| Process many URLs per store in a loop | Bypass site Terms of Service |
| Save JSON output for downstream Store DNA work | Build embeddings (that's the next step) |

> **Note:** Google Maps and Facebook pages are often JavaScript-heavy or login-protected. For production, pair this pattern with official APIs or headless browsers where needed.

## 1. Setup

Install dependencies (run once), then configure Azure OpenAI credentials.

Copy `.env.example` to `.env` in the project root and fill in your keys.

In [ ]:
# Run once if packages are missing
# %pip install -r ../requirements.txt

In [ ]:
import json
import os
import re
import smtplib
from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from openai import AzureOpenAI

# Optional HTML extractors — notebook degrades gracefully if missing
try:
    import trafilatura
except ImportError:
    trafilatura = None

try:
    from readability import Document
except ImportError:
    Document = None

# Load secrets from project-root .env
PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "data" / "external"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Configuration

Azure OpenAI is used to convert scraped text into structured store attributes.
SMTP settings are optional — only needed if you want email alerts.

In [ ]:
# --- Azure OpenAI ---
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5.2")

# --- Optional email alerts ---
SMTP_USER = os.getenv("SMTP_USER")
SMTP_PASS = os.getenv("SMTP_PASS")

# HTTP headers — mimics a browser so simple pages return HTML instead of blocking us
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; StoreDNA-Extractor/1.0)"}

# Max characters sent to the LLM per page (keeps token cost predictable)
MAX_TEXT_CHARS = 12_000

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

## 3. Input validation

Basic guards before we hit the network or call the LLM.

In [ ]:
def validate_url(url: str) -> None:
    """Raise ValueError if the URL is not a valid http(s) link."""
    if not re.match(r"^https?://", url.strip()):
        raise ValueError(f"Invalid URL: {url}")


def validate_email(email: str) -> None:
    """Raise ValueError if email is provided but malformed."""
    if email and not re.match(r"^[^@]+@[^@]+\.[^@]+$", email):
        raise ValueError(f"Invalid email: {email}")

## 4. Scrape — fetch and clean page text

Given any public URL, we:
1. Download the HTML with `requests`
2. Extract readable text using **trafilatura** (best), **readability** (fallback), or raw `<p>` tags (last resort)
3. Capture the page `<title>`

This is the core "scrape from the internet" step Sir referred to.

In [ ]:
def scrape(url: str) -> dict:
    """
    Fetch a public webpage and return cleaned text suitable for LLM extraction.

    Returns:
        dict with keys: url, title, text, scraped_at
    """
    validate_url(url)

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    html = response.text
    text = None

    # Strategy 1: trafilatura — strong at articles and content-heavy pages
    if trafilatura:
        text = trafilatura.extract(html)

    # Strategy 2: readability — pulls main content block from noisy HTML
    if not text and Document:
        doc = Document(html)
        text = re.sub(r"<[^>]+>", " ", doc.summary())

    # Strategy 3: raw paragraph tags — works on very simple pages
    if not text:
        paragraphs = re.findall(
            r"<p[^>]*>(.*?)</p>", html, re.DOTALL | re.IGNORECASE
        )
        text = " ".join(paragraphs)
        text = re.sub(r"<[^>]+>", " ", text)

    # Page title from <title> tag
    title_match = re.search(
        r"<title[^>]*>(.*?)</title>", html, re.DOTALL | re.IGNORECASE
    )
    title = title_match.group(1).strip() if title_match else "Untitled"

    return {
        "url": url,
        "title": title,
        "text": (text or "")[:MAX_TEXT_CHARS],
        "scraped_at": datetime.now(timezone.utc).isoformat(),
    }

## 5. Analyze — LLM extraction for Store DNA attributes

The original script extracted **AI news** fields. For StoreDNA we ask the LLM to pull **store-level signals** that can later feed embeddings and business recommendations.

Examples of what we extract:
- Location, format, services
- Review sentiment themes
- Promotions and competitive signals
- Local market keywords

In [ ]:
STORE_DNA_JSON_SCHEMA = """
{
  "store_name": "",
  "retailer": "",
  "address": "",
  "city": "",
  "state": "",
  "zip_code": "",
  "store_format": "",
  "services": [],
  "hours": "",
  "review_sentiment": "positive|neutral|negative|unknown",
  "review_themes": [],
  "promotions_mentioned": [],
  "local_keywords": [],
  "competitive_signals": [],
  "summary": "",
  "confidence": "high|medium|low"
}
"""


def _strip_json_fences(content: str) -> str:
    """Remove markdown code fences the model sometimes wraps around JSON."""
    content = re.sub(r"^```json\s*", "", content.strip())
    content = re.sub(r"\s*```$", "", content)
    return content.strip()


def analyze_for_store_dna(
    article: dict,
    store_id: str = "",
    retailer: str = "",
    source_type: str = "",
) -> dict:
    """
    Use Azure OpenAI to extract structured store attributes from scraped page text.

    Args:
        article: output from scrape()
        store_id: internal store identifier (context for the LLM)
        retailer: retailer name (context for the LLM)
        source_type: e.g. retailer_page, reviews, social, news
    """
    prompt = f"""
You are building data for a Retail Store DNA product.
Extract store-level attributes from the webpage content below.

Context:
- store_id: {store_id or "unknown"}
- retailer: {retailer or "unknown"}
- source_type: {source_type or "unknown"}
- page_url: {article["url"]}
- page_title: {article["title"]}

Rules:
- Return ONLY valid JSON matching the schema below.
- Use empty strings or empty lists when information is not present.
- Do not invent facts; set confidence to "low" if the page has little store info.
- review_sentiment should reflect customer reviews if this is a reviews page.

JSON schema:
{STORE_DNA_JSON_SCHEMA}

Page content:
{article["text"]}
"""

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )

    content = _strip_json_fences(response.choices[0].message.content)
    parsed = json.loads(content)

    # Attach provenance so we always know where each signal came from
    parsed["_meta"] = {
        "store_id": store_id,
        "retailer": retailer,
        "source_type": source_type,
        "url": article["url"],
        "page_title": article["title"],
        "scraped_at": article["scraped_at"],
    }

    return parsed

## 6. Optional — email alert

Same pattern as the original script. Skipped automatically if SMTP credentials are not set.

In [ ]:
def send_email(receiver: str, analysis: dict) -> bool:
    """Send a plain-text summary email. Returns False if skipped or misconfigured."""
    if not receiver:
        return False
    if not SMTP_USER or not SMTP_PASS:
        return False

    validate_email(receiver)

    subject = analysis.get("store_name") or analysis.get("summary", "Store DNA extraction")
    key_points = analysis.get("review_themes") or analysis.get("competitive_signals") or []

    body = (
        f"Store: {analysis.get('store_name', 'N/A')}\n"
        f"Retailer: {analysis.get('retailer', 'N/A')}\n\n"
        f"Summary:\n{analysis.get('summary', '')}\n\n"
        f"Signals:\n" + "\n".join(f"- {x}" for x in key_points)
    )

    msg = MIMEMultipart()
    msg["Subject"] = str(subject)[:120]
    msg["From"] = SMTP_USER
    msg["To"] = receiver
    msg.attach(MIMEText(body, "plain"))

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(SMTP_USER, SMTP_PASS)
        server.send_message(msg)

    return True

## 7. Pipeline — scrape + analyze for one URL

Combines the two core steps. This is what you run per URL.

In [ ]:
def extract_store_signals(
    url: str,
    store_id: str = "",
    retailer: str = "",
    source_type: str = "",
    email: str = "",
) -> dict:
    """
    End-to-end: validate → scrape → LLM extract → optional email.

    Returns structured JSON ready to save or merge into a store profile.
    """
    validate_url(url)
    validate_email(email)

    article = scrape(url)
    analysis = analyze_for_store_dna(
        article,
        store_id=store_id,
        retailer=retailer,
        source_type=source_type,
    )

    email_sent = send_email(email, analysis)

    return {
        **analysis,
        "email_sent": email_sent,
    }

## 8. Quick test — single URL

Run this cell with **one real store URL** to verify scraping and LLM extraction work before batching.

Replace the URL below with a retailer store page, Yelp listing, or other public page.

In [ ]:
TEST_URL = "https://www.walmart.com/store/1-rogers-ar"

single_result = extract_store_signals(
    url=TEST_URL,
    store_id="WMT-0001",
    retailer="Walmart",
    source_type="retailer_page",
)

print(json.dumps(single_result, indent=2))

## 9. Batch processing — many stores, many URLs

Load a CSV with one row per URL. Expected columns:

| Column | Description |
|--------|-------------|
| `store_id` | Your internal store key |
| `retailer` | Retailer name |
| `source_type` | `retailer_page`, `reviews`, `social`, `news`, etc. |
| `url` | Public page to scrape |

Results are grouped by `store_id` and saved to `data/external/`.

In [ ]:
def load_store_urls(csv_path: Path) -> pd.DataFrame:
    """Load and validate the store URL inventory CSV."""
    df = pd.read_csv(csv_path)
    required = {"store_id", "retailer", "source_type", "url"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")
    return df


def run_batch_extraction(
    csv_path: Path,
    email: str = "",
) -> dict[str, list[dict]]:
    """
    Scrape and analyze every URL in the CSV.

    Returns:
        dict mapping store_id → list of extraction results (one per URL)
    """
    df = load_store_urls(csv_path)
    store_results: dict[str, list[dict]] = {}

    for _, row in df.iterrows():
        store_id = str(row["store_id"])
        url = str(row["url"]).strip()

        print(f"Processing {store_id} | {row['source_type']} | {url}")

        try:
            result = extract_store_signals(
                url=url,
                store_id=store_id,
                retailer=str(row["retailer"]),
                source_type=str(row["source_type"]),
                email=email,
            )
            store_results.setdefault(store_id, []).append(result)
        except Exception as exc:
            # Keep going — one bad URL should not stop the whole batch
            error_record = {
                "error": str(exc),
                "_meta": {
                    "store_id": store_id,
                    "url": url,
                    "source_type": str(row["source_type"]),
                },
            }
            store_results.setdefault(store_id, []).append(error_record)
            print(f"  ⚠ Failed: {exc}")

    return store_results


def save_store_profiles(store_results: dict[str, list[dict]], output_dir: Path) -> None:
    """Write one JSON file per store_id for downstream embedding work."""
    output_dir.mkdir(parents=True, exist_ok=True)

    for store_id, records in store_results.items():
        profile = {
            "store_id": store_id,
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "sources": records,
        }
        out_path = output_dir / f"{store_id}.json"
        out_path.write_text(json.dumps(profile, indent=2), encoding="utf-8")
        print(f"Saved {out_path}")

In [ ]:
CSV_PATH = PROJECT_ROOT / "data" / "stores_sample.csv"

batch_results = run_batch_extraction(CSV_PATH)
save_store_profiles(batch_results, OUTPUT_DIR)

print(f"\nDone. {len(batch_results)} store profile(s) written to {OUTPUT_DIR}")

## 10. Inspect saved output

Each file under `data/external/` is a **store profile** — all scraped sources merged for one store. This is the input to the next StoreDNA step (embedding + peer matching).

In [ ]:
saved_files = sorted(OUTPUT_DIR.glob("*.json"))

for path in saved_files:
    print(f"\n{'=' * 60}\n{path.name}\n{'=' * 60}")
    print(path.read_text(encoding="utf-8")[:2000])

## 11. Next steps for Store DNA

1. **Expand URL inventory** — add Google Maps, Facebook, Yelp, local news URLs per store in `data/stores_sample.csv`.
2. **Blend internal data** — merge POS, foot traffic, planogram, and promo data (not covered by this notebook).
3. **Build embeddings** — concatenate or embed each store profile text → store vector (the "DNA").
4. **Downstream analytics** — clustering, peer-store matching, gap analysis, revenue recommendations.

---

### Source types you can add to the CSV

| `source_type` | Example URL |
|---------------|-------------|
| `retailer_page` | `walmart.com/store/...` |
| `reviews` | Yelp / Google reviews public page |
| `social` | Public Facebook business page |
| `news` | Local news mentioning the store |
| `jobs` | Indeed listing near the store (hiring signal) |